# Intervals.icu + Nightscout Analysis

This notebook loads your **latest activity** from Intervals.icu and aligns it with glucose values from Nightscout.

It then computes interval-based glucose summaries and charts the activity window.

## 1) Setup

Install dependencies if needed:

```bash
pip install requests pandas matplotlib python-dateutil
```

In [ ]:
! pip install requests pandas matplotlib python-dateutil


In [ ]:
import os
import hashlib
from datetime import datetime, timedelta, timezone

import requests
import pandas as pd
import matplotlib.pyplot as plt
from dateutil import parser

pd.set_option('display.max_columns', 200)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
os.environ["INTERVALS_ATHLETE_ID"] = "i298161"
os.environ["INTERVALS_API_KEY"]="59inwnqxsjucv0pp5gnbwunh"
os.environ["NIGHTSCOUT_BASE_URL"]="https://p01--nightscout--jfv4bxh9hrff.code.run/"

## 2) Configuration

You can set values directly below or provide environment variables.

In [ ]:
# Intervals.icu
INTERVALS_BASE_URL = os.getenv('INTERVALS_BASE_URL', 'https://intervals.icu')
INTERVALS_ATHLETE_ID = os.getenv('INTERVALS_ATHLETE_ID', '0')  # '0' means: the authenticated athlete
INTERVALS_API_KEY = os.getenv('INTERVALS_API_KEY', '')

# Nightscout
NIGHTSCOUT_BASE_URL = os.getenv('NIGHTSCOUT_BASE_URL', '').rstrip('/')
NIGHTSCOUT_TOKEN = os.getenv('NIGHTSCOUT_TOKEN', '')  # preferred when available
NIGHTSCOUT_API_SECRET = os.getenv('NIGHTSCOUT_API_SECRET', '')  # fallback if token is not used

# Analysis window and binning
PRE_ACTIVITY_HOURS = int(os.getenv('PRE_ACTIVITY_HOURS', '2'))
POST_ACTIVITY_HOURS = int(os.getenv('POST_ACTIVITY_HOURS', '6'))
BIN_MINUTES = int(os.getenv('BIN_MINUTES', '15'))

if not INTERVALS_API_KEY:
    raise ValueError('Set INTERVALS_API_KEY env var (Intervals.icu personal API key).')
if not NIGHTSCOUT_BASE_URL:
    raise ValueError('Set NIGHTSCOUT_BASE_URL env var (e.g. https://your-site.example.com).')
# Optional direct streams endpoint override (for Bearer-style access)
INTERVALS_STREAMS_BASE_URL = os.getenv('INTERVALS_STREAMS_BASE_URL', '').rstrip('/')
INTERVALS_AUTH_MODE = os.getenv('INTERVALS_AUTH_MODE', 'basic').lower()  # 'basic' or 'bearer'



In [ ]:
# Removed stray token that caused a SyntaxError.


## 3) API Helpers

In [ ]:
def intervals_get(path, params=None):
    url = f"{INTERVALS_BASE_URL}{path}"
    response = requests.get(url, params=params or {}, auth=('API_KEY', INTERVALS_API_KEY), timeout=30)
    response.raise_for_status()
    return response.json()


def nightscout_auth_headers():
    if NIGHTSCOUT_TOKEN:
        return {}
    if NIGHTSCOUT_API_SECRET:
        hashed = hashlib.sha1(NIGHTSCOUT_API_SECRET.encode('utf-8')).hexdigest()
        return {'API-SECRET': hashed}
    return {}


def nightscout_get(path, params=None):
    params = dict(params or {})
    if NIGHTSCOUT_TOKEN:
        params['token'] = NIGHTSCOUT_TOKEN

    url = f"{NIGHTSCOUT_BASE_URL}{path}"
    response = requests.get(url, params=params, headers=nightscout_auth_headers(), timeout=30)
    response.raise_for_status()
    return response.json()

## 4) Get Your Latest Intervals.icu Activity

In [ ]:
today = datetime.now().date()
oldest = (today - timedelta(days=60)).isoformat()
newest = (today + timedelta(days=1)).isoformat()

activities = intervals_get(
    f'/api/v1/athlete/{INTERVALS_ATHLETE_ID}/activities',
    params={'oldest': oldest, 'newest': newest}
)

if not activities:
    raise ValueError('No activities returned by Intervals.icu in the selected date window.')

activities_df = pd.DataFrame(activities)
activities_df['start_date_local'] = pd.to_datetime(activities_df['start_date_local'])
latest_activity = activities_df.sort_values('start_date_local').iloc[-1].to_dict()
latest_activity

In [ ]:
activity_id = latest_activity['id']
activity_start_local = pd.to_datetime(latest_activity['start_date_local'])

# Use elapsed_time if present, otherwise moving_time, else default 1 hour
duration_seconds = latest_activity.get('elapsed_time') or latest_activity.get('moving_time') or 3600
activity_end_local = activity_start_local + pd.to_timedelta(int(duration_seconds), unit='s')

window_start_local = activity_start_local - pd.to_timedelta(PRE_ACTIVITY_HOURS, unit='h')
window_end_local = activity_end_local + pd.to_timedelta(POST_ACTIVITY_HOURS, unit='h')

print(f"Latest activity: {latest_activity.get('name', latest_activity.get('type', 'Unknown'))} ({activity_id})")
print(f"Activity local start: {activity_start_local}")
print(f"Activity local end:   {activity_end_local}")
print(f"Query window start:   {window_start_local}")
print(f"Query window end:     {window_end_local}")

## 5) Fetch Glucose Data from Nightscout

In [ ]:
# Nightscout's entries API supports dateString filters
query_params = {
    'count': 20000,
    'find[dateString][$gte]': window_start_local.isoformat(),
    'find[dateString][$lte]': window_end_local.isoformat(),
}

entries = nightscout_get('/api/v1/entries/sgv.json', params=query_params)
if not entries:
    raise ValueError('No Nightscout glucose entries returned in this range.')

glucose_df = pd.DataFrame(entries)

# 'date' is epoch milliseconds in most Nightscout deployments
if 'date' in glucose_df.columns:
    glucose_df['timestamp'] = pd.to_datetime(glucose_df['date'], unit='ms', utc=True).dt.tz_convert(None)
elif 'dateString' in glucose_df.columns:
    glucose_df['timestamp'] = pd.to_datetime(glucose_df['dateString'])
else:
    raise ValueError('Nightscout entries do not include date or dateString.')

if 'sgv' not in glucose_df.columns:
    raise ValueError("Nightscout entries are missing 'sgv' values.")

glucose_df = glucose_df[['timestamp', 'sgv']].dropna().sort_values('timestamp').reset_index(drop=True)
glucose_df.head()

## 6) Build Interval-Based Summary

In [ ]:
analysis_df = glucose_df.copy()
analysis_df = analysis_df[(analysis_df['timestamp'] >= window_start_local) & (analysis_df['timestamp'] <= window_end_local)]
analysis_df = analysis_df.set_index('timestamp')

interval_summary = analysis_df['sgv'].resample(f'{BIN_MINUTES}min').agg(['count', 'mean', 'median', 'min', 'max', 'std'])
interval_summary = interval_summary.reset_index()

interval_summary.head(20)

In [ ]:
summary_stats = {
    'activity_id': activity_id,
    'activity_start': activity_start_local,
    'activity_end': activity_end_local,
    'glucose_points': int(analysis_df['sgv'].shape[0]),
    'avg_glucose': float(analysis_df['sgv'].mean()),
    'min_glucose': float(analysis_df['sgv'].min()),
    'max_glucose': float(analysis_df['sgv'].max()),
}
pd.Series(summary_stats)

## 7) Plot Glucose with Activity Overlay

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(glucose_df['timestamp'], glucose_df['sgv'], marker='o', markersize=2, linewidth=1, label='Glucose (SGV)')
ax.axvspan(activity_start_local, activity_end_local, alpha=0.2, label='Activity window')
ax.set_title('Nightscout Glucose Around Latest Intervals.icu Activity')
ax.set_xlabel('Time')
ax.set_ylabel('Glucose (mg/dL)')
ax.legend()
plt.tight_layout()
plt.show()

## 8) Save Outputs (Optional)

In [ ]:
out_dir = 'output'
os.makedirs(out_dir, exist_ok=True)

glucose_df.to_csv(f'{out_dir}/glucose_raw_latest_activity.csv', index=False)
interval_summary.to_csv(f'{out_dir}/glucose_interval_summary_latest_activity.csv', index=False)
activities_df.sort_values('start_date_local', ascending=False).head(1).to_csv(
    f'{out_dir}/latest_activity_metadata.csv', index=False
)

print('Saved CSV files to ./output')

## Notes

- Intervals.icu uses basic auth with username `API_KEY` and password = your API key.
- Nightscout supports token-based access (`?token=...`) and API secret auth (SHA1 hash in `API-SECRET` header).
- Verify timezone behavior in your own data if you need strict local-time alignment.

## 9) HR and BG Correlation Around Activity

This section fetches activity streams from Intervals.icu, aligns HR with Nightscout BG, and computes correlation metrics.

In [ ]:
print('HR_FETCH_CELL_VERSION=2026-03-23-v3')
import numpy as np


def get_intervals_streams_for_activity(activity_id, requested_streams='time,heartrate,power'):
    candidates = []

    # Build URL candidates
    if INTERVALS_STREAMS_BASE_URL:
        try:
            candidates.append(('url', INTERVALS_STREAMS_BASE_URL.format(activity_id=activity_id)))
        except Exception:
            candidates.append(('url', INTERVALS_STREAMS_BASE_URL))

    candidates.extend([
        ('url', f'{INTERVALS_BASE_URL}/api/v1/activity/{activity_id}/streams.json'),
        ('url', f'{INTERVALS_BASE_URL}/api/v1/activities/{activity_id}/streams'),
        ('path', f'/api/v1/activity/{activity_id}/streams.json'),
        ('path', f'/api/v1/activities/{activity_id}/streams'),
    ])

    # Try both param naming styles used by stream endpoints
    param_variants = [
        {'streams': requested_streams},
        {'types': requested_streams},
        {'streams': requested_streams, 'types': requested_streams},
        {},
    ]

    # Auth strategies
    auth_variants = [
        ('bearer', {'Authorization': f'Bearer {INTERVALS_API_KEY}'}, None),
        ('basic', {}, ('API_KEY', INTERVALS_API_KEY)),
    ]

    attempts = []

    for target_type, target in candidates:
        for params in param_variants:
            for auth_name, headers, basic_auth in auth_variants:
                try:
                    if target_type == 'url':
                        resp = requests.get(target, headers=headers, params=params, auth=basic_auth, timeout=20)
                        resp.raise_for_status()
                        data = resp.json()
                    else:
                        # Use helper for path-based requests only on basic auth
                        if auth_name != 'basic':
                            continue
                        data = intervals_get(target, params=params)

                    return data, {'target': target, 'params': params, 'auth': auth_name}
                except Exception as e:
                    attempts.append(f"{target} | {params} | {auth_name}: {type(e).__name__}")

    debug_tail = '\n'.join(attempts[-8:])
    raise ValueError('Unable to fetch streams after multiple endpoint/auth/param attempts. Last attempts:\n' + debug_tail)


def _stream_values(stream_obj):
    if stream_obj is None:
        return None
    if isinstance(stream_obj, list):
        return stream_obj
    if isinstance(stream_obj, dict):
        if isinstance(stream_obj.get('data'), list):
            return stream_obj.get('data')
        if isinstance(stream_obj.get('values'), list):
            return stream_obj.get('values')
    return None


def _normalize_streams(raw):
    out = {}

    if isinstance(raw, list):
        for item in raw:
            if isinstance(item, dict):
                k = item.get('type') or item.get('name')
                v = _stream_values(item)
                if k and isinstance(v, list):
                    out[str(k)] = v
        return out

    if isinstance(raw, dict):
        if isinstance(raw.get('streams'), list):
            out.update(_normalize_streams(raw['streams']))
        for k, v in raw.items():
            values = _stream_values(v)
            if isinstance(values, list):
                out[str(k)] = values

    return out


raw_streams, fetch_meta = get_intervals_streams_for_activity(activity_id, requested_streams='time,heartrate,power')
print('Stream fetch used:', fetch_meta)

streams_map = _normalize_streams(raw_streams)
if not streams_map:
    raise ValueError('No streams returned for this activity.')

print('Available stream keys:', sorted(streams_map.keys()))

hr_keys = ['heartrate', 'hr', 'heart_rate', 'heartRate']
time_keys = ['time', 'seconds', 'sec']
power_keys = ['power', 'watts', 'watt']

hr_vals = next((streams_map[k] for k in hr_keys if k in streams_map and streams_map[k]), None)
time_vals = next((streams_map[k] for k in time_keys if k in streams_map and streams_map[k]), None)
power_vals = next((streams_map[k] for k in power_keys if k in streams_map and streams_map[k]), None)

if not hr_vals:
    raise ValueError('No heart rate data found for this activity. See printed stream keys above.')

if not time_vals:
    time_vals = list(range(len(hr_vals)))

n = min(len(hr_vals), len(time_vals))
if power_vals:
    n = min(n, len(power_vals))

activity_hr_df = pd.DataFrame({
    'timestamp': activity_start_local + pd.to_timedelta(time_vals[:n], unit='s'),
    'hr': pd.to_numeric(pd.Series(hr_vals[:n]), errors='coerce')
})

if power_vals:
    activity_hr_df['power'] = pd.to_numeric(pd.Series(power_vals[:n]), errors='coerce')
else:
    activity_hr_df['power'] = np.nan

activity_hr_df = activity_hr_df.dropna(subset=['hr'])
print(f'Retrieved {len(activity_hr_df)} HR data points.')
print('First 10 HR values:', activity_hr_df['hr'].head(10).tolist())

activity_hr_df = activity_hr_df[
    (activity_hr_df['timestamp'] >= activity_start_local) &
    (activity_hr_df['timestamp'] <= activity_end_local)
]

bg_for_merge = glucose_df.rename(columns={'sgv': 'bg'}).sort_values('timestamp')[['timestamp', 'bg']]
aligned_df = pd.merge_asof(
    activity_hr_df.sort_values('timestamp'),
    bg_for_merge,
    on='timestamp',
    direction='nearest',
    tolerance=pd.Timedelta('3min')
).dropna(subset=['bg'])

if aligned_df.empty:
    raise ValueError('No overlapping HR/BG samples after alignment. Try a larger merge tolerance than 3min.')

aligned_df.head()





In [ ]:
# Define high-intensity by top HR quantile within this activity
HI_QUANTILE = float(os.getenv('HI_QUANTILE', '0.85'))
hr_hi_threshold = aligned_df['hr'].quantile(HI_QUANTILE)
aligned_df['high_intensity'] = aligned_df['hr'] >= hr_hi_threshold

corr_pearson = aligned_df['hr'].corr(aligned_df['bg'], method='pearson')
corr_spearman = aligned_df['hr'].corr(aligned_df['bg'], method='spearman')

hi_df = aligned_df[aligned_df['high_intensity']]
corr_hi_pearson = hi_df['hr'].corr(hi_df['bg'], method='pearson') if len(hi_df) > 2 else np.nan

# Correlation of binary high-intensity flag vs BG (point-biserial equivalent via Pearson)
corr_high_flag_bg = aligned_df['high_intensity'].astype(int).corr(aligned_df['bg'], method='pearson')

if 'power' in aligned_df.columns and aligned_df['power'].notna().sum() > 2:
    corr_power_bg = aligned_df['power'].corr(aligned_df['bg'], method='pearson')
else:
    corr_power_bg = np.nan

# Lag scan: positive lag means BG is shifted later (BG response after HR change)
per_min = aligned_df.set_index('timestamp')[['hr', 'bg']].resample('1min').mean().interpolate(limit_direction='both')
lag_results = []
for lag in range(-60, 61, 5):
    shifted = per_min['bg'].shift(-lag)
    c = per_min['hr'].corr(shifted)
    lag_results.append({'lag_minutes': lag, 'pearson_hr_bg': c})
lag_corr_df = pd.DataFrame(lag_results)

best_lag_row = lag_corr_df.iloc[lag_corr_df['pearson_hr_bg'].abs().idxmax()]

corr_summary = pd.Series({
    'samples_aligned': int(len(aligned_df)),
    'high_intensity_hr_threshold': float(hr_hi_threshold),
    'pearson_hr_bg': float(corr_pearson) if pd.notna(corr_pearson) else np.nan,
    'spearman_hr_bg': float(corr_spearman) if pd.notna(corr_spearman) else np.nan,
    'pearson_hr_bg_high_intensity_only': float(corr_hi_pearson) if pd.notna(corr_hi_pearson) else np.nan,
    'pearson_high_intensity_flag_vs_bg': float(corr_high_flag_bg) if pd.notna(corr_high_flag_bg) else np.nan,
    'pearson_power_bg': float(corr_power_bg) if pd.notna(corr_power_bg) else np.nan,
    'best_abs_lag_minutes': int(best_lag_row['lag_minutes']),
    'best_abs_lag_corr': float(best_lag_row['pearson_hr_bg']) if pd.notna(best_lag_row['pearson_hr_bg']) else np.nan,
})
corr_summary



In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 5))

ax1.plot(aligned_df['timestamp'], aligned_df['hr'], color='tab:red', linewidth=1.2, label='Heart Rate')
ax1.set_xlabel('Time')
ax1.set_ylabel('Heart Rate (bpm)', color='tab:red')
ax1.tick_params(axis='y', labelcolor='tab:red')

# Highlight high-intensity timestamps
ax1.scatter(
    aligned_df.loc[aligned_df['high_intensity'], 'timestamp'],
    aligned_df.loc[aligned_df['high_intensity'], 'hr'],
    color='darkred', s=8, alpha=0.7, label='High-intensity HR'
)

ax2 = ax1.twinx()
ax2.plot(aligned_df['timestamp'], aligned_df['bg'], color='tab:blue', linewidth=1.2, label='BG (SGV)')
ax2.set_ylabel('Glucose (mg/dL)', color='tab:blue')
ax2.tick_params(axis='y', labelcolor='tab:blue')

ax1.set_title('Heart Rate and Blood Glucose During Latest Activity')

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left')

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lag_corr_df['lag_minutes'], lag_corr_df['pearson_hr_bg'], marker='o', linewidth=1)
ax.axhline(0, color='black', linewidth=0.8)
best_lag_label = "Best lag: {} min".format(int(best_lag_row['lag_minutes']))
ax.axvline(best_lag_row['lag_minutes'], color='tab:green', linestyle='--', linewidth=1, label=best_lag_label)
ax.set_title('Lagged Correlation: HR vs BG')
ax.set_xlabel('Lag (minutes, + means BG delayed)')
ax.set_ylabel('Pearson correlation')
ax.legend()
plt.tight_layout()
plt.show()
